[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_02/SFM_ch2_distribution_comparison/SFM_ch2_distribution_comparison.ipynb)

# SFM_ch2_distribution_comparison

Comprehensive Distribution Comparison for Financial Returns
Description:
- Fit Normal, Student-t, GED, and Stable to S&P 500 returns
- PDF overlay on histogram (log scale)
- 2x2 QQ-plot panel for all distributions
- Tail CDF comparison (survival function, log-log)
- AIC/BIC model selection ranking
Statistics of Financial Markets course

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats
from scipy.stats import levy_stable, gennorm
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Chart style settings - Nature journal quality
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 8
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['legend.fontsize'] = 8
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['lines.linewidth'] = 0.75

# Color palette
MAIN_BLUE = '#1A3A6E'
CRIMSON = '#DC3545'
FOREST = '#2E7D32'
AMBER = '#B5853F'
ORANGE = '#E67E22'

# Output directory
CHART_DIR = os.path.join('..', '..', '..', 'charts')
os.makedirs(CHART_DIR, exist_ok=True)

def save_fig(name):
    """Save figure with transparent background."""
    plt.savefig(os.path.join(CHART_DIR, f'{name}.pdf'),
                bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(CHART_DIR, f'{name}.png'),
                bbox_inches='tight', transparent=True, dpi=300)
    plt.close()
    print(f"   Saved: {name}.pdf/.png")

## Download Data and Fit All Distributions

In [ ]:
# Download S&P 500 data
data = yf.download('^GSPC', start='2000-01-01', end='2025-01-01', progress=False)
close = data['Close'].squeeze()
log_ret = np.log(close / close.shift(1)).dropna()
returns = log_ret.values

print(f"   Period: {close.index[0].strftime('%Y-%m-%d')} to {close.index[-1].strftime('%Y-%m-%d')}")
print(f"   Observations: {len(returns)}")
print(f"   Mean: {returns.mean():.6f}")
print(f"   Std:  {returns.std():.6f}")
print(f"   Skew: {stats.skew(returns):.4f}")
print(f"   Kurt: {stats.kurtosis(returns, fisher=True):.4f}")
print()

# --- Fit Normal ---
norm_mu, norm_sigma = stats.norm.fit(returns)
print(f"   Normal fit:")
print(f"     mu    = {norm_mu:.6f}")
print(f"     sigma = {norm_sigma:.6f}")

# --- Fit Student-t ---
t_df, t_loc, t_scale = stats.t.fit(returns)
print(f"   Student-t fit:")
print(f"     df    = {t_df:.4f}")
print(f"     loc   = {t_loc:.6f}")
print(f"     scale = {t_scale:.6f}")

# --- Fit GED (Generalized Error Distribution = gennorm in scipy) ---
ged_beta, ged_loc, ged_scale = gennorm.fit(returns)
print(f"   GED fit:")
print(f"     beta  = {ged_beta:.4f}")
print(f"     loc   = {ged_loc:.6f}")
print(f"     scale = {ged_scale:.6f}")

# --- Fit Stable ---
pars_init = levy_stable._fitstart(returns)
stable_params = levy_stable.fit(returns, floc=pars_init[2], fscale=pars_init[3])
s_alpha, s_beta, s_loc, s_scale = stable_params
print(f"   Stable fit:")
print(f"     alpha = {s_alpha:.4f}")
print(f"     beta  = {s_beta:.4f}")
print(f"     loc   = {s_loc:.6f}")
print(f"     scale = {s_scale:.6f}")
print()

# --- Log-likelihoods, AIC, BIC ---
n = len(returns)

ll_norm = np.sum(stats.norm.logpdf(returns, norm_mu, norm_sigma))
ll_t = np.sum(stats.t.logpdf(returns, t_df, loc=t_loc, scale=t_scale))
ll_ged = np.sum(gennorm.logpdf(returns, ged_beta, loc=ged_loc, scale=ged_scale))
ll_stable = np.sum(levy_stable.logpdf(returns, s_alpha, s_beta,
                                       loc=s_loc, scale=s_scale))

def aic_bic(ll, k, n):
    aic = -2 * ll + 2 * k
    bic = -2 * ll + k * np.log(n)
    return aic, bic

aic_norm, bic_norm = aic_bic(ll_norm, 2, n)
aic_t, bic_t = aic_bic(ll_t, 3, n)
aic_ged, bic_ged = aic_bic(ll_ged, 3, n)
aic_stable, bic_stable = aic_bic(ll_stable, 4, n)

# Comparison table
print("   Model Selection Comparison:")
print(f"   {'Distribution':>12s}  {'LogLik':>12s}  {'AIC':>12s}  {'BIC':>12s}  {'#Params':>7s}")
print(f"   {'-'*12}  {'-'*12}  {'-'*12}  {'-'*12}  {'-'*7}")

models = [
    ('Normal', ll_norm, aic_norm, bic_norm, 2),
    ('Student-t', ll_t, aic_t, bic_t, 3),
    ('GED', ll_ged, aic_ged, bic_ged, 3),
    ('Stable', ll_stable, aic_stable, bic_stable, 4),
]

# Sort by AIC
models_sorted = sorted(models, key=lambda x: x[2])
for name, ll, aic, bic, k in models_sorted:
    print(f"   {name:>12s}  {ll:>12.2f}  {aic:>12.2f}  {bic:>12.2f}  {k:>7d}")

## PDF Overlay Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

# Histogram
counts, bins, _ = ax.hist(returns, bins=150, density=True, alpha=0.45,
                           color='gray', edgecolor='white', linewidth=0.3,
                           label='S&P 500 log-returns')

# PDF overlay grid
x_plot = np.linspace(returns.min() * 1.1, returns.max() * 1.1, 1000)

# Normal (dashed, MAIN_BLUE)
pdf_norm = stats.norm.pdf(x_plot, norm_mu, norm_sigma)
ax.plot(x_plot, pdf_norm, color=MAIN_BLUE, linewidth=1.2, linestyle='--',
        label=f'Normal (AIC={aic_norm:.0f})')

# Student-t (solid, CRIMSON)
pdf_t = stats.t.pdf(x_plot, t_df, loc=t_loc, scale=t_scale)
ax.plot(x_plot, pdf_t, color=CRIMSON, linewidth=1.3, linestyle='-',
        label=f'Student-t (AIC={aic_t:.0f})')

# GED (solid, FOREST)
pdf_ged = gennorm.pdf(x_plot, ged_beta, loc=ged_loc, scale=ged_scale)
ax.plot(x_plot, pdf_ged, color=FOREST, linewidth=1.3, linestyle='-',
        label=f'GED (AIC={aic_ged:.0f})')

# Stable (solid, ORANGE)
pdf_stable = levy_stable.pdf(x_plot, s_alpha, s_beta,
                              loc=s_loc, scale=s_scale)
ax.plot(x_plot, pdf_stable, color=ORANGE, linewidth=1.3, linestyle='-',
        label=f'Stable (AIC={aic_stable:.0f})')

ax.set_yscale('log')
ax.set_ylim(1e-2, None)
ax.set_xlabel('Log-return')
ax.set_ylabel('Density (log scale)')
ax.set_title('Ch.2: Distribution Comparison — PDF Overlay (S&P 500)',
             fontweight='bold', fontsize=10)

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3,
          frameon=False, fontsize=7)

plt.tight_layout()
save_fig('ch2_dist_comparison_overlay')

## QQ-Plot Panel

In [ ]:
sorted_returns = np.sort(returns)
n = len(sorted_returns)
empirical_quantiles = (np.arange(1, n + 1) - 0.5) / n

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

# --- Normal QQ ---
ax = axes[0, 0]
theo_q = stats.norm.ppf(empirical_quantiles, norm_mu, norm_sigma)
ax.scatter(theo_q, sorted_returns, s=1, alpha=0.3, color=MAIN_BLUE, rasterized=True)
q_min, q_max = min(theo_q.min(), sorted_returns.min()), max(theo_q.max(), sorted_returns.max())
ax.plot([q_min, q_max], [q_min, q_max], color=CRIMSON, linewidth=0.8, linestyle='--')
ax.set_xlabel('Normal theoretical quantiles')
ax.set_ylabel('Sample quantiles')
ax.set_title('Normal', fontweight='bold', fontsize=9)

# --- Student-t QQ ---
ax = axes[0, 1]
theo_q = stats.t.ppf(empirical_quantiles, t_df, loc=t_loc, scale=t_scale)
ax.scatter(theo_q, sorted_returns, s=1, alpha=0.3, color=CRIMSON, rasterized=True)
q_min, q_max = min(theo_q.min(), sorted_returns.min()), max(theo_q.max(), sorted_returns.max())
ax.plot([q_min, q_max], [q_min, q_max], color=MAIN_BLUE, linewidth=0.8, linestyle='--')
ax.set_xlabel('Student-t theoretical quantiles')
ax.set_ylabel('Sample quantiles')
ax.set_title(f'Student-t (df={t_df:.2f})', fontweight='bold', fontsize=9)

# --- GED QQ ---
ax = axes[1, 0]
theo_q = gennorm.ppf(empirical_quantiles, ged_beta, loc=ged_loc, scale=ged_scale)
ax.scatter(theo_q, sorted_returns, s=1, alpha=0.3, color=FOREST, rasterized=True)
q_min, q_max = min(theo_q.min(), sorted_returns.min()), max(theo_q.max(), sorted_returns.max())
ax.plot([q_min, q_max], [q_min, q_max], color=CRIMSON, linewidth=0.8, linestyle='--')
ax.set_xlabel('GED theoretical quantiles')
ax.set_ylabel('Sample quantiles')
ax.set_title(f'GED ($\\beta$={ged_beta:.2f})', fontweight='bold', fontsize=9)

# --- Stable QQ ---
ax = axes[1, 1]
theo_q = levy_stable.ppf(empirical_quantiles, s_alpha, s_beta,
                          loc=s_loc, scale=s_scale)
ax.scatter(theo_q, sorted_returns, s=1, alpha=0.3, color=ORANGE, rasterized=True)
q_min, q_max = min(theo_q.min(), sorted_returns.min()), max(theo_q.max(), sorted_returns.max())
ax.plot([q_min, q_max], [q_min, q_max], color=CRIMSON, linewidth=0.8, linestyle='--')
ax.set_xlabel('Stable theoretical quantiles')
ax.set_ylabel('Sample quantiles')
ax.set_title(f'Stable ($\\alpha$={s_alpha:.2f})', fontweight='bold', fontsize=9)

fig.suptitle('Ch.2: QQ-Plot Panel — S&P 500 Log-Returns',
             fontweight='bold', fontsize=11, y=1.02)

plt.tight_layout()
save_fig('ch2_dist_comparison_qq')

## Tail Comparison

In [ ]:
fig, (ax_right, ax_left) = plt.subplots(1, 2, figsize=(12, 4.5))

sorted_ret = np.sort(returns)
n = len(sorted_ret)

# ---- Right tail: P(X > x) ----
# Empirical survival function
right_vals = sorted_ret[sorted_ret > 0]
right_sf_emp = 1.0 - np.arange(1, len(right_vals) + 1) / n  # P(X > x_i)

ax_right.scatter(right_vals, right_sf_emp, s=2, alpha=0.2, color='gray',
                 rasterized=True, label='Empirical', zorder=1)

# Theoretical right-tail survival curves
x_right = np.linspace(1e-4, sorted_ret.max() * 1.05, 1000)
x_right = x_right[x_right > 0]

sf_norm = stats.norm.sf(x_right, norm_mu, norm_sigma)
sf_t = stats.t.sf(x_right, t_df, loc=t_loc, scale=t_scale)
sf_ged = gennorm.sf(x_right, ged_beta, loc=ged_loc, scale=ged_scale)
sf_stable = levy_stable.sf(x_right, s_alpha, s_beta,
                            loc=s_loc, scale=s_scale)

ax_right.plot(x_right, sf_norm, color=MAIN_BLUE, linewidth=1.2, linestyle='--',
              label='Normal', zorder=2)
ax_right.plot(x_right, sf_t, color=CRIMSON, linewidth=1.2, linestyle='-',
              label='Student-t', zorder=2)
ax_right.plot(x_right, sf_ged, color=FOREST, linewidth=1.2, linestyle='-',
              label='GED', zorder=2)
ax_right.plot(x_right, sf_stable, color=ORANGE, linewidth=1.2, linestyle='-',
              label='Stable', zorder=2)

# 3-sigma and 5-sigma markers
sigma_emp = returns.std()
for sigma_mult, lbl in [(3, '$3\\sigma$'), (5, '$5\\sigma$')]:
    xv = sigma_mult * sigma_emp
    ax_right.axvline(xv, color=AMBER, linewidth=0.7, linestyle='--', alpha=0.7)
    ax_right.text(xv * 1.05, 0.5, lbl, fontsize=7, color=AMBER,
                  transform=ax_right.get_xaxis_transform(), va='center')

ax_right.set_xscale('log')
ax_right.set_yscale('log')
ax_right.set_xlabel('$x$')
ax_right.set_ylabel('$P(X > x)$')
ax_right.set_title('Right Tail Survival', fontweight='bold', fontsize=9)
ax_right.legend(loc='lower left', fontsize=7, frameon=False)

# ---- Left tail: P(X < -x) for x > 0 ----
left_vals = np.abs(sorted_ret[sorted_ret < 0])[::-1]  # |x| for negative returns, ascending
left_cdf_emp = np.arange(1, len(left_vals) + 1) / n  # P(X < -x_i)
# We want P(X < -x) as a function of x>0, so reverse
left_x = left_vals
left_sf = 1.0 - left_cdf_emp  # survival of |negative returns|
# Actually: P(X < -x) = CDF(-x)
neg_sorted = np.sort(-returns[returns < 0])  # positive values, ascending
left_sf_emp = 1.0 - np.arange(1, len(neg_sorted) + 1) / n

ax_left.scatter(neg_sorted, left_sf_emp, s=2, alpha=0.2, color='gray',
                rasterized=True, label='Empirical', zorder=1)

# Theoretical left-tail: P(X < -x) = CDF(-x)
x_left = np.linspace(1e-4, np.abs(sorted_ret.min()) * 1.05, 1000)
x_left = x_left[x_left > 0]

cdf_norm_left = stats.norm.cdf(-x_left, norm_mu, norm_sigma)
cdf_t_left = stats.t.cdf(-x_left, t_df, loc=t_loc, scale=t_scale)
cdf_ged_left = gennorm.cdf(-x_left, ged_beta, loc=ged_loc, scale=ged_scale)
cdf_stable_left = levy_stable.cdf(-x_left, s_alpha, s_beta,
                                    loc=s_loc, scale=s_scale)

ax_left.plot(x_left, cdf_norm_left, color=MAIN_BLUE, linewidth=1.2, linestyle='--',
             label='Normal', zorder=2)
ax_left.plot(x_left, cdf_t_left, color=CRIMSON, linewidth=1.2, linestyle='-',
             label='Student-t', zorder=2)
ax_left.plot(x_left, cdf_ged_left, color=FOREST, linewidth=1.2, linestyle='-',
             label='GED', zorder=2)
ax_left.plot(x_left, cdf_stable_left, color=ORANGE, linewidth=1.2, linestyle='-',
             label='Stable', zorder=2)

# 3-sigma and 5-sigma markers
for sigma_mult, lbl in [(3, '$3\\sigma$'), (5, '$5\\sigma$')]:
    xv = sigma_mult * sigma_emp
    ax_left.axvline(xv, color=AMBER, linewidth=0.7, linestyle='--', alpha=0.7)
    ax_left.text(xv * 1.05, 0.5, lbl, fontsize=7, color=AMBER,
                 transform=ax_left.get_xaxis_transform(), va='center')

ax_left.set_xscale('log')
ax_left.set_yscale('log')
ax_left.set_xlabel('$|x|$')
ax_left.set_ylabel('$P(X < -x)$')
ax_left.set_title('Left Tail CDF', fontweight='bold', fontsize=9)
ax_left.legend(loc='lower left', fontsize=7, frameon=False)

fig.suptitle('Ch.2: Tail CDF Comparison — S&P 500 Log-Returns (log-log)',
             fontweight='bold', fontsize=10, y=1.02)

plt.tight_layout()
save_fig('ch2_dist_comparison_tails')